# PixelPlay word-sync model: singing fine-tune (Colab)
Runtime → Change runtime type → **T4 GPU**. Put this folder (`tools/alignment/singing_finetune`) and your `data/` folder in Google Drive under `MyDrive/pixelplay-singing/`, laid out as in README.md.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/pixelplay-singing/singing_finetune
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!pip -q install -r requirements.txt

## Data manifests

In [ ]:
!python prepare_data.py utterances --root ../data/GTSinger/English --out ../data/manifests/gtsinger
!python prepare_data.py longform --audio-dir ../data/CSD/english/wav --lyrics-dir ../data/CSD/english/txt --out ../data/manifests/csd

## Baseline (what the app does today)

In [ ]:
!python evaluate_alignment.py --jamendo ../data/jamendolyrics --models facebook/wav2vec2-base-960h --out ../baseline.json

## Train (checkpoints are written to Drive, so a disconnect loses at most 500 steps: just run this cell again and it resumes)

In [ ]:
!python train.py --train ../data/manifests/gtsinger/train.jsonl ../data/manifests/csd/train.jsonl \
  --val ../data/manifests/gtsinger/val.jsonl ../data/manifests/csd/val.jsonl \
  --accomp-dir ../data/accompaniment --out ../runs/singing-v1 --steps 6000 --workers 2 --resume

## Export + compare

In [ ]:
!python export_onnx.py --checkpoint ../runs/singing-v1/best --out ../runs/singing-v1/model_int8.onnx
!python evaluate_alignment.py --jamendo ../data/jamendolyrics --models facebook/wav2vec2-base-960h ../runs/singing-v1/best ../runs/singing-v1/model_int8.onnx --out ../runs/singing-v1/alignment_report.json